# 어텐션 메커니즘 구현하기 목차
* [Chapter 1 개요](#chapter1)
* [Chapter 2 긴 시퀸스 모델링의 문제점](#chapter2)
* [Chapter 3 어텐션 메커니즘으로 데이터 의존성 포착하기](#chapter3)
* [Chapter 4 셀프 어텐션으로 입력의 서로 다른 부분에 주의 기울이기](#chapter4)
    * [Section 4-1 훈련 가능한 가중치가 없는 간단한 셀프 어텐션 메커니즘](#section4-1)
    * [Section 4-2 모든 입력 토큰에 대해 어텐션 가중치 계산하기](#section4-2)
* [Chapter 5 훈련 가능한 가중치를 가진 셀프 어텐션 구현하기](#chapter5)
    * [Section 5-1 단계별로 어텐션 가중치 계산하기](#section5-1)
    * [Section 5-2 셀프 어텐션 파이썬 클래스 구현하기](#section5-2)
* [Chapter 6 코잘 어텐션으로 미래의 단어를 감추기](#chapter6)
    * [Section 6-1 코자리 어텐션으로 마스크 적용하기](#section6-1)
    * [Section 6-2 드롭아웃으로 어텐션 가중치에 추가적으로 마스킹하기](#section6-2)    
    * [Section 6-3 코잘 어텐션 클래스 구현하기](#section6-3)       
* [Chapter 7 싱글 헤드 어텐션을 멀티헤드 어텐션으로 확장하기](#chapter7)     

## Chapter 1 개요 <a class="anchor" id="chapter1"></a>
1. 어텐션 메커니즘을 살펴보고 내부가 어떻게 동작하는지 알아봅니다.

2. 셀프 어텐션 메커니즘을 둘러싼 다른 부분을 구현하여 작동 방식을 알아본다.

    ![구현단계](image/03-00-process2.png)

3. 네 가지 버전의 어텐션 메커니즘을 구현한다.
    - 이전에 구현한 것에 새로운 기능을 추가하는 식으로 만든다.

        ![구현단계](image/03-00-process3.png)




## Chapter 2 긴 시퀸스 모델링의 문제점 <a class="anchor" id="chapter2"></a>
1. LMM 시대 이전 어텐션 메커니즘을 사용하지 않던 구조가 가졌던 문제점
    - 한 언어에서 다른 언어로 텍스트를 번역하는 언어 모델을 만든다고 가정
    - 소스 언어와 타켓 언어가 문법 구조가 다르기 때문에 한 단어씩 번역할 수 없다.
    - 이 문제를 해결하기 위해 DNN에서 인코더와 디코더 2개의 서브모듈을 사용한다.
        - 인코더: 소스 언어 문장을 벡터로 인코딩
        - 디코더: 벡터를 타켓 언어 문장으로 디코딩

        ![번역](image/04-01-attention.png)    

    - 트랜스포머가 개발되기 전에는 순환 신경망(RNN recurrent neural network)을 사용했다.
        - 시퀸스 데이터를 처리하는 데 특화된 구조
        - 이전 시점의 출력을 현재 시점의 입력으로 사용하는 순환 구조를 가짐
        - 인코더가 입력 텍스트를 받아 순차적으로 처리
        - 인코더는 각 단계마다 은닉 상태를 업데이트 하며, 최종 은닉 상태로 입력 시퀸스 전체 의미를 포착
           - 인코더가 전체 입력 텍스트를 하나의 은닉 상태로 처리한다.
        - 디코더도 매 스텝마다 은닉 상태를 업데이트하며 이를 통해 다음 단어 예측에 필요한 문맥정보를 다음 스텝에 전달
           - 디코더가 은닉 상태를 받아 출력을 생성한다.
        - 은닉 상태를 임베딩 벡터 개념으로 생각할 수 있다.
        - 디코딩 단계에서 RNN이 이전 은닉 상태를 참조할 수 없다.
        - RNN은 긴 시퀸스를 처리하는 데 어려움이 있다.
           - 최종 은닉 상태에만 의존하게 되어 맥락을 놓칠 수 있다.
           - 멀리 떨어진 단어에 의존성이 있는 복잡한 문장의 경우는 특히 그렇다.

            ![RNN](image/04-01-RNN.png)    

## Chapter 3 어텐션 메커니즘으로 데이터 의존성 포착하기 <a class="anchor" id="chapter3"></a>
1. RNN이 인코딩된 전체 입력을 하나의 은닉 상태에 저장해서 디코더에 전달하는 문제를 해결하기 위해 바흐다나우 어텐션 메커니즘이 개발되었다.

2. 바흐다나우 어텐션
    - 인코더-디코더 RNN을 수정하여 디코딩 단계마다 디코더가 선택적으로 입력 시퀸스의 서로 다른 부분을 참조할 수 있다.
    - 디코더가 매 스텝마다 인코더의 모든 은닉 상태를 참조할 수 있다.
    - 3년 후에 RNN 구조가 자연어 처리를 위한 심층 신경망을 구축하는데 필수적이지 않다고 밝혀졌다.

        ![RNN](image/03-02-battention.png)    


3. 트랜스포머 모델이 등장했고, 바흐나나우 어텐션에 영감을 받은 셀프 어텐션이 포함되었다.
    - 셀프 어텐션은 입력 시퀸스의 서로 다른 위치 간의 의존성을 포착하는 메커니즘
    - 트랜스포머 모델은 RNN을 사용하지 않고도 긴 시퀸스를 효과적으로 처리할 수 있다.
    - 트랜스포머 모델은 자연어 처리 분야에서 혁신을 일으켰고, 이후 대형 언어 모델(LLM)의 발전에 중요한 역할을 했다.

4. 셀프 어텐션은 트랜스포머에서 사용되는 메커니즘
    - 입력 시퀸스에 있는 각 위치가 동일 시퀸스에 있는 다른 모든 위치와 상호작용하여 중요도를 부여한다.

## Chapter 4 셀프 어텐션으로 입력의 서로 다른 부분에 주의 기울이기 <a class="anchor" id="chapter4"></a>
1. 셀프 어텐션은 트랜스포머 구조를 바탕으로 하는 모든 LLM의 기반이 된다.
    - 셀프 어텐션의 '셀프'는 하나의 입력 시퀸스에 있는 서로 다른 위치의 원소 사이에서 어텐션 가중치를 계산한다.
    - 문장 안의 단어와 이미지 안에 있는 픽셀 사이의 관계와 의존성을 평가하고 학습한다.
    - 2 개의 다른 시퀸스에 있는 원소 사이의 관계에 초점을 맞추는 전통적인 어텐션 메커니즘과는 다르다.



### Section 4-1 훈련 가능한 가중치가 없는 간단한 셀프 어텐션 메커니즘 <a class="anchor" id="section4-1"></a>
1. 목표는 훈련 가능한 가중치를 추가하기 전에 셀프 어텐션에 있는 몇 가지 핵심 개념을 이해하는 것이다.
    - 셀프 어텐션의 목표는 다른 모든 입력 원소의 정보를 조합하여 각각의 입력 원소에 대한 문맥 벡터를 계산하는 것이다.

2. "Your journey starts with one step"이라는 입력 텍스트
    - x<sup>(1)</sup>에 해당하는 시퀴스의 각 원소는 "your"를 표현하는 d 차원 임베딩 벡터이다. 
    - 셀프 어텐션에서는 입력 시퀴스에 있는 각 원소 x<sup>(i)</sup>에 대한 문맥 벡터 z<sup>(i)</sup>를 계산하는 것이 목표이다.
       - 문맥 벡터는 정보가 풍부한 임베딩 벡터로 생각할 수 있다.
    - 두 번재 입력 원소 "journey" x<sup>(2)</sup>에 대한 임베딩 벡터와 문맥 벡터 z<sup>(2)</sup>를 살펴보면.
       - 문맥 벡터 z<sup>(2)</sup>는  x<sup>(2)</sup>와 다른 모든 입력 (x<sup>(1)</sup>~x<sup>(i)</sup>) 사이의 정보를 담은 임베딩이다.

3. 문맥 벡터는 셀프 어텐션에서 매우 중요한 역할을 한다.
    - 문맥 벡터의 목적은 입력 시퀸스에 있는 다른 모든 원소의 정보를 통합해 이 시퀸스에 있는 각 원소의 표현을 풍부하게 만드는 것이다.
    - LLM에서 문장에서 다른 단어 사이의 관계와 관련성을 이해하는 것이 필요하다

        ![셀프 어텐션](image/03-03-attenction.png)  

4. 셀프 어텐션을 구하는 첫 번째 단계는 어텐션 점수(attention score)를 계산하는 것이다.
    - 어텐션 점수는 입력 시퀸스에 있는 서로 다른 원소 사이의 관련성을 측정한다.
    - 어텐션 점수는 두 입력 원소 x<sup>(i)</sup>와 x<sup>(j)</sup> 사이의 유사성을 나타낸다.
    - 두 번째 입력 원소  x<sup>2</sup>를 쿼리로 사용하여 문맥 벡터 z<sup>(2)</sup>를 계산한다
    - 쿼리 x<sup>(2)</sup>와 다른 모든 원소 사이의 점곱(dot product)을 계산하여 어텐션 점수를 w를 구한다.
    - 점곱은 두 벡터 사이의 유사성을 측정하는 간단한 방법이다.
       - 점곱이 클수록 두 벡터가 얼마나 가까이 놓여 있는지 정량화 할 수 있다.
    - 셀프 어텐션에서 점곱은 시퀸스에 있는 각 원소가 다른 소에 얼마나 관련이 있는지 평가하는 데 사용된다.

        ![셀프 어텐션 점수](image/03-03-score.png)  

In [35]:
import torch

# 입력 시퀸스 (6개의 단어, 각 단어는 3차원 벡터로 표현 - 단어 임베딩)
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

# 쿼리 토큰과 각 입력 토큰 사이의 어텐션 점수를 점 곱으로 계산
query = inputs[1] # 두 번째 입력 토큰을 쿼리 토큰으로 사용 (journey)
atten_scores_2 = torch.empty(inputs.shape[0]) # 어텐션 점수를 저장할 텐서

# 각 입력 토큰에 대해 쿼리 토큰과의 점 곱 계산
for i, x_i in enumerate(inputs):
    atten_scores_2[i] = torch.dot(x_i, query)
    
print(atten_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


5. 다음 단계로 어텐션 점수를 정규화한다.
    - 정규화를 하는 목적은 어텐션 가중치의 합이 1이 되도록 하는 것이다.
    - 정규화를 하면 해석이 용이하고 LMM을 훈련할 때 안정성을 유지하는데 도움이 된다.

6. 입력 쿼리 x<sup>(2)</sup>에 대한 어텐션 가중치 ω<sub>21</sub>에서  ω<sub>2T</sub>까지 구한다.
    - 다음 단계는 어텐션 점수를 정규화하여 어텐션 가중치 α<sub>21</sub>에서  α<sub>2T</sub>까지 구하는 것이다.

        ![가중치](image/03-03-weight.png)

In [36]:
atten_weights_2_tmp = atten_scores_2 / atten_scores_2.sum()  # 어텐션 점수를 정규화하여 어텐션 가중치 계산
print("어텐션 가중치: ", atten_weights_2_tmp)
print("합: ", atten_weights_2_tmp.sum())

어텐션 가중치:  tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
합:  tensor(1.0000)


7. 일반적으로 소프트맥스 함수를 사용하여 정규화를 진행한다.
   - 어텐션 가중치가 항상 양수가 되도록 보장한다.
   - 가중치가 높을 수록 중요도가 높다.
   - 큰 입력아니 매우 작은 입력을 처리 할 때 오버플로나 언더플로 같은 수치 불안정 문제 발생할 수 있다.
   - 실전에 광법위하게 성능 최적화가 된 파이토치의 소프트맥스 함수를 사용한다.

In [37]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum()

atten_weights_2_naive = softmax_naive(atten_scores_2)
print("어텐션 가중치: ", atten_weights_2_naive)
print("합: ", atten_weights_2_naive.sum())

어텐션 가중치:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
합:  tensor(1.)


In [38]:
atten_weights_2 = torch.softmax(atten_scores_2, dim=0)
print("어텐션 가중치: ", atten_weights_2)
print("합: ", atten_weights_2.sum())

어텐션 가중치:  tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
합:  tensor(1.)


8. 임베딩된 입력 토큰 x<sup>(i)</sup>와 각 토큰에 해당하는 어턴션 가중치를 곱한 후 모두 더해서 문맥 벡터 z<sup>(i)</sup>를 계산한다.
    - 문맥 벡터 z<sup>(i)</sup>는 입력 쿼리 x<sup>(i)</sup>와 시퀸스에 있는 다른 모든 원소 사이의 정보를 통합한 것이다.
    - 문맥 벡터 z<sup>(i)</sup>는 입력 쿼리 x<sup>(i)</sup>와 시퀸스에 있는 다른 모든 원소 사이의 관련성을 반영한다.

        ![문맥 벡터](image/03-03-context2.png)

In [39]:
query = inputs[1] # 두 번째 입력 토큰을 쿼리 토큰으로 사용 (journey)
context_vec2 = torch.zeros(inputs.shape[1]) # 문맥 벡터를 저장할 텐서
for i, x_i in enumerate(inputs):
    context_vec2 += atten_weights_2[i] * x_i
print("문맥 벡터: ", context_vec2)

문맥 벡터:  tensor([0.4419, 0.6515, 0.5683])


### Section 4-2 모든 입력 토큰에 대해 어텐션 가중치 계산하기 <a class="anchor" id="section4-2"></a>
1. 지금까지 노란 색으로 강조된 부분에 대한 어텐션 가중치와 문맥 벡터를 계산했다.

    ![어텐션 가중치](image/03-03-weight2.png)

2. 두 번째 원소 z<sub>2</sub>뿐만 아니라 모든 문맥 벡터를 계산한다.

In [40]:
atten_scores = torch.empty(6, 6) # 어텐션 점수를 저장할 텐서

# 모든 입력 토큰에 대해 쿼리 토큰과의 점 곱 계산
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        atten_scores[i, j] = torch.dot(x_i, x_j)
print(atten_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [41]:
# for 루프는 느리기 때문에 행렬 곱셈을 사용한다.
attn_scores = inputs @ inputs.T # 행렬 곱셈
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [42]:
# 정규화 진행
attn_weights = torch.softmax(attn_scores, dim=1) # 1: 마지막 차원을 기준으로 정규화 진행
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [43]:
# 각 행의 값을 더해 1이 되는지 확인
row_2_sum = attn_weights[2].sum()
print("세 번째 행의 합: ", row_2_sum)

# 모든 행의 합이 1인지 확인
all_rows_sum = attn_weights.sum(dim=-1)
print("모든 행의 합: ", all_rows_sum)

세 번째 행의 합:  tensor(1.0000)
모든 행의 합:  tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])


In [44]:
# 어텐션 가중치와 입력 행렬을 곱해서 문맥 벡터 계산하기
all_context_vecs = attn_weights @ inputs
print("모든 문맥 벡터:\n", all_context_vecs)

모든 문맥 벡터:
 tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])


In [45]:
# 두 번째 문맥 벡터와 비교
print("두 번째 문맥 벡터:\n", context_vec2)

두 번째 문맥 벡터:
 tensor([0.4419, 0.6515, 0.5683])


## Chapter 5 훈련 가능한 가중치를 가진 셀프 어텐션 구현하기 <a class="anchor" id="chapter5"></a>
1. 훈련 가능한 가중치를 가진 셀프 어텐션 메커니즘은 스케일드 점곱 어텐션이라고 부른다.
    - 특정 입력 원소에 대한 입력 벡터의 가중치 합으로 문맥 벡터를 계산한다.
    - 모델 훈련 과정에서 업데이트되는 훈련 가능한 가중치 행렬이 추가된다.
        - 가중치 행렬을 통해 모데이 '좋은' 문맥 벡터를 생성하는 방법을 학습한다. 

### Section 5-1 단계별로 어텐션 가중치 계산하기 <a class="anchor" id="section5-1"></a>
1. 훈련 가능한 가중치 행렬 3개, 즉 W<sub>q</sub>, W<sub>k</sub>, W<sub>v</sub>를 추가하여 셀프 어텐션 메커니즘을 단계별로 구현한다.
    - 3개의 행렬을 사용해 임베딩된 입력 토큰 x<sup>(i)</sup>를 각각 쿼리(q<sup>(i)</sup>), 키(k<sup>(i)</sup>), 값(v<sup>(i)</sup>) 벡터로 투영한다.
    - 입력 원소 x에 대한 쿼리(q), 키(k), 값(v) 벡터 계산
    - 두 번째 입력 원소 x<sup>(2)</sup>를 쿼리의 입력으로 사용 
    - 쿼리 벡터는 입력과 행렬 W를 곱해서 계산한다.

        ![단계별 가중치](image/03-04-stepWeight.png)

2. 문맥 벡터 z<sup>(2)</sup>를 구한 후 코드를 수정하여 모든 문맥 벡터를 구한다.

    ![테이블](image/03-04-table3-2.png)

In [7]:
import torch

# 입력 시퀸스 (6개의 단어, 각 단어는 3차원 벡터로 표현 - 단어 임베딩)
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

x_2 = inputs[1] # 두 번째 입력 토큰 (journey)
d_in = inputs.shape[1] # 입력 임베딩의 크기, 3
d_out = 2 # 출력 임베딩의 크기, 2

torch.manual_seed(123)
W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False) # 쿼리 가중치 행렬
W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False) # 키 가중치 행렬
W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False) # 값 가중치 행렬
#W_query = torch.tensor([[-2.0832,2.4916],[0.306,-1.9368],[0.395,0.395]]) # 쿼리 가중치 행렬
#W_key = torch.tensor([[0.326,1.8413],[-1.0984,1.8216],[-1.4147,1.1726]]) # 키 가중치 행렬
#W_value = torch.tensor([[0.7324,-1.504],[-0.4526,0.7108],[-0.5131,1.0748]]) # 값 가중치 행렬

print("쿼리 가중치 행렬:\n", W_query)
print("키 가중치 행렬:\n", W_key)
print("값 가중치 행렬:\n", W_value)

쿼리 가중치 행렬:
 Parameter containing:
tensor([[0.2961, 0.5166],
        [0.2517, 0.6886],
        [0.0740, 0.8665]])
키 가중치 행렬:
 Parameter containing:
tensor([[0.1366, 0.1025],
        [0.1841, 0.7264],
        [0.3153, 0.6871]])
값 가중치 행렬:
 Parameter containing:
tensor([[0.0756, 0.1966],
        [0.3164, 0.4017],
        [0.1186, 0.8274]])


In [8]:
# 쿼리, 키, 값 벡터 계산
query_2 = x_2 @ W_query
key_2 = x_2 @ W_key
value_2 = x_2 @ W_value

# 쿼리의 출력 결과는 2차원 벡터
print("입력 벡터: ", x_2)
print("쿼리 벡터: ", query_2)
print("키 벡터: ", key_2)
print("값 벡터: ", value_2)

입력 벡터:  tensor([0.5500, 0.8700, 0.6600])
쿼리 벡터:  tensor([0.4306, 1.4551])
키 벡터:  tensor([0.4433, 1.1419])
값 벡터:  tensor([0.3951, 1.0037])


In [9]:
# 키, 값을 편하게 가져오도록 미리 계산
keys = inputs @ W_key   # 모든 입력 토큰에 대한 키 벡터 계산
values = inputs @ W_value # 모든 입력 토큰에 대한 값 벡터 계산
print("모든 키 벡터:\n", keys, keys.shape)
print("모든 값 벡터:\n", values, values.shape)


모든 키 벡터:
 tensor([[0.3669, 0.7646],
        [0.4433, 1.1419],
        [0.4361, 1.1156],
        [0.2408, 0.6706],
        [0.1827, 0.3292],
        [0.3275, 0.9642]]) torch.Size([6, 2])
모든 값 벡터:
 tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]]) torch.Size([6, 2])


3. 각각의 가중치 행렬로 입력을 변환한 쿼리와 키 벡터 사이의 점곱을 계산하여 어텐션 점수를 구한다.
    - 두 번째 입력원소("journey")의 쿼리 벡터와 모든 입력 원소의 키 벡터 사이의 점곱을 계산하여 어텐션 점수를 구한다.

    ![점곱 계산](image/03-04-step2.png)

In [11]:
# 어텐션 점수 W22 계산
keys_2 = keys[1] # 두 번째 입력 토큰에 대한 키 벡터
print("두 번째 키 벡터: ", keys_2, keys_2.shape)

print("query_2: ",query_2)
attn_scores_22 = query_2.dot(keys_2)
print("어텐션 점수 W22: ", attn_scores_22)


두 번째 키 벡터:  tensor([0.4433, 1.1419]) torch.Size([2])
query_2:  tensor([0.4306, 1.4551])
어텐션 점수 W22:  tensor(1.8524)


In [12]:
# 행렬 곱셈으로 일반화하여 모든 어텐션 점수 계산
atten_score_2 = query_2 @ keys.T

# 두 번째 원소가 앞서 계산한 attn_scores_22와 동일하다. 3.2369
print("모든 어텐션 점수:\n", atten_score_2)


모든 어텐션 점수:
 tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


4. 어텐션 점수에서 어텐션 가중치를 구한다.
    - 소프트맥스 함수를 사용하여 어텐션 점수를 정규화한다.
    - 어텐션 점수를 키의 임베딩 차원의 제곱근으로 나눈다.
       - 제곱근은 0.5를 제곱하는 것과 수학적으로 동일하다.
    - 임베딩 차원 크기로 정규화를 하는 이유는 그레이디언트가 작아지는 것을 피하여 성능을 향상시키기 위해서이다.
    - 임베딩 차원이 커지면 소프트맥스 함수 때문에 역전파 과정에서 매우 작은 그레이디언트를 생성할 수 있다.
    - 임베딩 차원을 제곱근으로 나누기때문에 셀프 어텐션 메커니즘을 스케일드 점곱 어텐션이라고 부른다.

        ![어텐션 가중치](image/03-04-step3.png)

In [15]:
d_k = keys.shape[-1] # 키 벡터의 차원
print("키 벡터의 차원: ", d_k)
attn_weights_2 = torch.softmax(atten_score_2 / (d_k ** 0.5), dim=-1)
print("어텐션 가중치: ", attn_weights_2)
print("합: ", attn_weights_2.sum())

키 벡터의 차원:  2
어텐션 가중치:  tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
합:  tensor(1.)


5. 문맥 벡터를 계산한다.
    - 모든 값 벡터를 어텐션 가중치를 통해 결합하여 문맥 벡터를 계산한다.

        ![문맥 계산](image/03-04-step4.png)

In [16]:
context_vec_2 = attn_weights_2 @ values
print("어텐션 가중치: ", attn_weights_2)
print("값 벡터:\n", values)
print("문맥 벡터: ", context_vec_2)

어텐션 가중치:  tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])
값 벡터:
 tensor([[0.1855, 0.8812],
        [0.3951, 1.0037],
        [0.3879, 0.9831],
        [0.2393, 0.5493],
        [0.1492, 0.3346],
        [0.3221, 0.7863]])
문맥 벡터:  tensor([0.3061, 0.8210])


### Section 5-2 셀프 어텐션 파이썬 클래스 구현하기 <a class="anchor" id="section5-2"></a>
1. 어텐션에서 사용할 쿼리 / 키 / 값 가중치를 초기화 한다.
    - W<sub>q</sub>, W<sub>k</sub>, W<sub>v</sub>는 훈련 가능한 가중치 행렬이다.

2. 각 가중치와 입력 토큰 임베딩(X)을 곱하여 쿼리, 키, 값 벡터를 계산한다.
    - 쿼리 벡터: Q = X @ W<sub>q</sub>
    - 키 벡터: K = X @ W<sub>k</sub>
    - 값 벡터: V = X @ W<sub>v</sub>

3. 쿼리 벡터와 키 벡터를 곱하여 어텐션 점수를 구한다.
    - 어텐션 점수: scores = Q @ K<sup>T</sup>
    - "Your journey starts with one step"
        - "Your"의 쿼리 벡터와  ["Your", "journey", "starts", "with", "one", "step"]의 키 벡터를 곱하여 어텐션 점수를 구한다.
        - "journey"의 쿼리 벡터와  ["Your", "journey", "starts", "with", "one", "step"]의 키 벡터를 곱하여 어텐션 점수를 구한다.
        - "starts"의 쿼리 벡터와  ["Your", "journey", "starts", "with", "one", "step"]의 키 벡터를 곱하여 어텐션 점수를 구한다.
        - "with"의 쿼리 벡터와  ["Your", "journey", "starts", "with", "one", "step"]의 키 벡터를 곱하여 어텐션 점수를 구한다.
        - "one"의 쿼리 벡터와  ["Your", "journey", "starts", "with", "one", "step"]의 키 벡터를 곱하여 어텐션 점수를 구한다.
        - "step"의 쿼리 벡터와  ["Your", "journey", "starts", "with", "one", "step"]의 키 벡터를 곱하여 어텐션 점수를 구한다.

4. 어텐션 점수를 정규화하여 어텐션 가중치를 구한다.
    - 어텐션 가중치: attn_weights = softmax(scores / sqrt(d_k))
    - d<sub>k</sub>는 키 벡터의 차원

5. 어텐션 가중치와 값 벡터를 곱하여 문맥 벡터를 계산한다.
    - 각각의 값 벡터와 어텐션 가중치를 곱한 후 모두 더해서 문맥 벡터를 계산한다.
    - 문맥 벡터: context_vec = attn_weights @ V
    
    ![전체](image/03-04-total.png)

In [20]:
import torch
import torch.nn as nn

# nn.Module을 상속받아 셀프 어텐션 클래스 정의
#   - 층을 생성하고 관리하기 위해 필요한 기능을 제공
class SelfAttention_v1(nn.Module):
    # 가중치 행렬의 크기를 랜덤하게 초기화
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False) # 쿼리 가중치 행렬
        self.W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False) # 키 가중치 행렬
        self.W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False) # 값 가중치 행렬
       
        
    def forward(self, x):
        # 입력 x에 대해 쿼리, 키, 값 벡터 계산
        queries = x @ self.W_query # 모든 입력 토큰에 대한 쿼리 벡터 계산
        #print("모든 쿼리 벡터:\n", queries)
        keys = x @ self.W_key     # 모든 입력 토큰에 대한 키 벡터 계산
        #print("모든 키 벡터:\n", keys)
        #print("모든 키 벡터.T:\n", keys.T)
        values = x @ self.W_value  # 모든 입력 토큰에 대한 값 벡터 계산
        attn_scores = queries @ keys.T # 모든 쿼리-키 쌍에 대한 어텐션 점수 계산
        #print("어텐션 점수 = (모든 쿼리 벡터 dot 모든 키 벡터):\n", attn_scores)
        attn_weights = torch.softmax(attn_scores / (keys.shape[-1] ** 0.5), dim=-1) # 어텐션 가중치 계산
        print("어텐션 가중치: ", attn_weights)
        context_vecs = attn_weights @ values # 문맥 벡터 계산
        return context_vecs

In [21]:
torch.manual_seed(123)  # 재현성을 위해 랜덤 시드 고정
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

sa_v1 = SelfAttention_v1(d_in=3, d_out=2) # 입력 임베딩 크기 3, 출력 임베딩 크기 2
print("문맥 벡터: ",sa_v1(inputs))

어텐션 가중치:  tensor([[0.1551, 0.2104, 0.2059, 0.1413, 0.1074, 0.1799],
        [0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
        [0.1503, 0.2256, 0.2192, 0.1315, 0.0914, 0.1819],
        [0.1591, 0.1994, 0.1962, 0.1477, 0.1206, 0.1769],
        [0.1610, 0.1949, 0.1923, 0.1501, 0.1265, 0.1752],
        [0.1557, 0.2092, 0.2048, 0.1419, 0.1089, 0.1794]])
문맥 벡터:  tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]])


6. 셀프 어텐션은 훈련 가능한 가중치 행렬을 사용한다.
    - W<sub>q</sub>, W<sub>k</sub>, W<sub>v</sub> 는 모델 훈련 과정에서 업데이트된다.

7. "SelfAttention_v1" 구현을 파이토치 nn.Liner를 사용하도록 개선한다.
    - nn.Linear는 입력과 가중치 행렬의 곱셈과 편향 추가를 자동으로 처리한다.
    - nn.Linear를 사용하여 쿼리, 키, 값 벡터를 계산하는 코드를 간소화한다.
    

In [59]:
# nn.Module을 상속받아 셀프 어텐션 클래스 정의
#   - 층을 생성하고 관리하기 위해 필요한 기능을 제공
class SelfAttention_v2(nn.Module):
    # 가중치 행렬의 크기를 랜덤하게 초기화
    def __init__(self, d_in, d_out, dkv_bias=False):
        super().__init__()
        #self.W_query = nn.Linear(d_in, d_out, bias=dkv_bias) # 쿼리 가중치 행렬
        #self.W_key = nn.Linear(d_in, d_out, bias=dkv_bias)   # 키 가중치 행렬
        #self.W_value = nn.Linear(d_in, d_out, bias=dkv_bias) # 값 가중치 행렬
        self.W_query = W_query; # 쿼리 가중치 행렬
        self.W_key = W_key;   # 키 가중치 행렬
        self.W_value = W_value; # 값 가중치 행렬
        
    def forward(self, x):
        # 입력 x에 대해 쿼리, 키, 값 벡터 계산
        queries = x @ self.W_query # 모든 입력 토큰에 대한 쿼리 벡터 계산
        keys = x @ self.W_key     # 모든 입력 토큰에 대한 키 벡터 계산
        values = x @ self.W_value  # 모든 입력 토큰에 대한 값 벡터 계산
        attn_scores = queries @ keys.T # 모든 쿼리-키 쌍에 대한 어텐션 점수 계산
        attn_weights = torch.softmax(attn_scores / (keys.shape[-1] ** 0.5), dim=-1) # 어텐션 가중치 계산
        context_vecs = attn_weights @ values # 문맥 벡터 계산
        return context_vecs

## Chapter 6 코잘 어텐션으로 미래의 단어를 감추기 <a class="anchor" id="chapter6"></a>
1. 마스크드 어텐션(Masked Attention)이라고 코잘 어텐션(Causal Attention)은 마스크드 어텐션의 한 형태입니다.
    - 주어진 토큰으로 어텐션 점수를 계산할 때 시퀸스의 이전 입력과 현재 입력만 참조하도록 한다.
    - 각 토큰을 처리할 대 입력 토큰에서 현재 토큰의 다음에 오는 미래 토큰을 마스킹한다.
    - 주대각선 위의 어텐션 가중치를 마스킹하고 마스킹하지 않은 어텐션 가중치를 정규하하여 각 행의 합이 1이 되도록 만든다.

        ![콰잘 어텐션](image/03-05-cousal.png)

### Section 6-1 코잘 어텐션으로 마스크 적용하기 <a class="anchor" id="section6-1"></a>
1. 어텐션 점수에 소프트맥수 함수를 적용하고, 주대각선 위의 원소를 0으로 만든다음, 각 행을 정규화 한다.

    ![단계](image/03-05-process.png)



In [60]:
# 소프트 맥스 함수를 사용하여 어텐션 가중치 계산
queries = inputs @ W_query # 모든 입력 토큰에 대한 쿼리 벡터 계산
keys = inputs @ W_key     # 모든 입력 토큰에 대한 키 벡터 계산
values = inputs @ W_value  # 모든 입력 토큰에 대한 값 벡터 계산
attn_scores = queries @ keys.T # 모든 쿼리-키 쌍에 대한 어텐션 점수 계산
print("어텐션 점수:\n", attn_scores)
attn_weights = torch.softmax(attn_scores / (keys.shape[-1] ** 0.5), dim=-1) # 어텐션 가중치 계산
print("어텐션 가중치: ", attn_weights)


어텐션 점수:
 tensor([[ 3.0275e+00,  4.6700e+00,  4.6156e+00,  2.6077e+00,  2.3363e+00,
          3.3023e+00],
        [ 6.8065e-01,  8.7638e-01,  8.4249e-01,  5.3906e-01, -5.2179e-03,
          8.9686e-01],
        [ 9.2223e-01,  1.2436e+00,  1.2047e+00,  7.4566e-01,  1.6454e-01,
          1.1652e+00],
        [-7.4484e-01, -1.2424e+00, -1.2407e+00, -6.6700e-01, -8.6060e-01,
         -7.2925e-01],
        [ 5.0179e+00,  7.5135e+00,  7.3950e+00,  4.2605e+00,  3.1792e+00,
          5.6752e+00],
        [-3.0058e+00, -4.6834e+00, -4.6352e+00, -2.6018e+00, -2.4628e+00,
         -3.2369e+00]])
어텐션 가중치:  tensor([[0.1016, 0.3247, 0.3124, 0.0755, 0.0623, 0.1234],
        [0.1679, 0.1928, 0.1883, 0.1519, 0.1034, 0.1957],
        [0.1631, 0.2047, 0.1991, 0.1440, 0.0954, 0.1937],
        [0.1853, 0.1303, 0.1305, 0.1958, 0.1707, 0.1874],
        [0.0682, 0.3984, 0.3663, 0.0399, 0.0186, 0.1086],
        [0.1898, 0.0580, 0.0600, 0.2525, 0.2786, 0.1612]])


In [61]:
# 파이토치 tril 함수로 주 대각선 위의 값이 0인 마스크 생성
context_length = attn_weights.shape[0]
print("컨텍스트 길이: ", context_length)

# 파이토치 tril 함수로 주 대각선 위의 값이 0인 마스크 생성
#   - 1로 채워진 (context_length, context_length) 크기의 행렬을 생성
make_simple = torch.tril(torch.ones(context_length, context_length))
print("마스크:\n", make_simple)

# 마스크와 어텐션 가중치를 곱하여 주대각선 위의 값을 0으로 만듦
make_simple = attn_weights * make_simple
print("마스크 적용 후 어텐션 가중치:\n", make_simple)


컨텍스트 길이:  6
마스크:
 tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])
마스크 적용 후 어텐션 가중치:
 tensor([[0.1016, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1679, 0.1928, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1631, 0.2047, 0.1991, 0.0000, 0.0000, 0.0000],
        [0.1853, 0.1303, 0.1305, 0.1958, 0.0000, 0.0000],
        [0.0682, 0.3984, 0.3663, 0.0399, 0.0186, 0.0000],
        [0.1898, 0.0580, 0.0600, 0.2525, 0.2786, 0.1612]])


In [62]:
# 어텐션 가중치를 합이 1이 되도록 정규화
row_sums = make_simple.sum(dim=-1, keepdim=True) # 각 행의 합 계산
print("각 행의 합:\n", row_sums)

make_simple = make_simple / row_sums # 각 행을 행의 합으로 나누어 정규화
print("정규화된 어텐션 가중치:\n", make_simple)

각 행의 합:
 tensor([[0.1016],
        [0.3608],
        [0.5669],
        [0.6419],
        [0.8914],
        [1.0000]])
정규화된 어텐션 가중치:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4655, 0.5345, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2877, 0.3611, 0.3513, 0.0000, 0.0000, 0.0000],
        [0.2887, 0.2030, 0.2033, 0.3050, 0.0000, 0.0000],
        [0.0765, 0.4469, 0.4110, 0.0448, 0.0209, 0.0000],
        [0.1898, 0.0580, 0.0600, 0.2525, 0.2786, 0.1612]])


2. 소프트맥스 함수의 수학적 성질을 사용해 마스킹된 어텐션 가중치를 더 작은 단계에서 효율적으로 계산할 수 있다.
    - 주대각선 위의 값을 음의 무한대로 마스킹한다.
    - 한 행에 음의 무한대 값이 있으면 소프트맥스 해당 값을 0으로 만듭니다.
    - 주대각선 위의 값을 1로 만들로 1을 음의 무한대 값으로 바꾸는 식으로 더 효율적인 마스킹 기법을 구현할 수 있다.

In [63]:
# 파이토치 triu 함수로 주 대각선 위의 값이 1인 마스크 생성
#   - 1로 채워진 (context_length, context_length) 크기의 행렬을 생성
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
print("마스크:\n", mask)

# 마스크를 사용하여 어텐션 점수에서 주 대각선 위의 값을 -inf로 만듦
#   - masked_fill: mask 행렬의 1인 위치에 -inf를 채움
#print("어텐션 점수:\n", attn_scores)
masked = attn_scores.masked_fill(mask == 1, float('-inf'))
#print("마스킹된 어텐션 점수:\n", masked)

# 소프트 맥스 적용
attn_weights = torch.softmax(masked / (keys.shape[-1] ** 0.5), dim=-1) # 어텐션 가중치 계산
print("마스킹된 어텐션 가중치:\n", attn_weights)

마스크:
 tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])
마스킹된 어텐션 가중치:
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4655, 0.5345, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2877, 0.3611, 0.3513, 0.0000, 0.0000, 0.0000],
        [0.2887, 0.2030, 0.2033, 0.3050, 0.0000, 0.0000],
        [0.0765, 0.4469, 0.4110, 0.0448, 0.0209, 0.0000],
        [0.1898, 0.0580, 0.0600, 0.2525, 0.2786, 0.1612]])


### Section 6-2 드롭아웃으로 어텐션 가중치에 추가적으로 마스킹하기 <a class="anchor" id="section6-2"></a>
1. 드롭아웃은 훈련 중에 은닉층의 유닛을 랜덤하게 선택해서 해당 유닛의 출력을 무시하는 기법니다.
    - 드롭아웃은 신경망이 특정 입력에 과도하게 적응하는 것을 방지하여 과대적합을 줄이는 데 도움이 된다.
    - 드롭아웃은 어텐션 가중치에 추가적인 마스킹을 적용하는 데 사용할 수 있다.
    - 어텐션 가중치를 계산한 후에 드롭아웃 마스크를 적용한다.

        ![드롭아웃](image/03-05-dropout.png)

In [64]:
torch.manual_seed(123)  # 재현성을 위해 랜덤 시드 고정
dropout = torch.nn.Dropout(p=0.5)  # 드롭아웃 확률 50%
example = torch.ones(6, 6)  # 6x6 크기의 행렬 생성
print("원본 행렬:\n", example)

# 드롭아웃 적용
#    - 원소 절반이 랜덤하게 0으로 바뀜
#    - 삭제 후 남은 원소들로 보상하기 위해 행렬에서 남은 원 값을 1 / 0.5 = 2 배로 늘린다.
#    - 입력과 출력의 크기를 동일하게 유지하기 위해서이다.
#    - 이런 보상은 전반적인 어텐션 가중치의 평균을 유지하는 데 중요하다.
dropped = dropout(example)  # 드롭아웃 적용
print("드롭아웃 적용 후 행렬:\n", dropped)

원본 행렬:
 tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])
드롭아웃 적용 후 행렬:
 tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [65]:
# 어텐션 가중치 행렬에 드롭아웃 적용
print(dropout(attn_weights))

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.7221, 0.7025, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.4066, 0.6100, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0417, 0.0000],
        [0.3796, 0.1159, 0.0000, 0.5051, 0.5572, 0.3223]])


### Section 6-3 코잘 어텐션 클래스 구현하기 <a class="anchor" id="section6-3"></a>
1. SelfAttention 클래스에 코잘 어텐션과 드롭아웃 기능 추가

2. 1개 이상의 입력으로 구성된 배치를 처리할 수 있도록 한다.

In [66]:
batch = torch.stack([inputs, inputs], dim=0)  # 2개의 입력으로 구성된 배치 생성
print("배치 입력:\n", batch, batch.shape)

배치 입력:
 tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]]) torch.Size([2, 6, 3])


In [68]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        
         # 가중치 고정을 위해 직접 텐서로 초기화
        self.W_query_tensor = torch.tensor([[-2.0832,2.4916],[0.306,-1.9368],[0.395,0.395]])
        self.W_key_tensor = torch.tensor([[0.326,1.8413],[-1.0984,1.8216],[-1.4147,1.1726]])
        self.W_value_tensor = torch.tensor([[0.7324,-1.504],[-0.4526,0.7108],[-0.5131,1.0748]])

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias) # 쿼리 가중치 행렬
        with torch.no_grad():
            self.W_query.weight.copy_(self.W_query_tensor[:, :d_out].T)
        
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias) # 키 가중치 행렬
        with torch.no_grad():
            self.W_key.weight.copy_(self.W_key_tensor[:, :d_out].T)
            
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias) # 값 가중치 행렬
        with torch.no_grad():
            self.W_value.weight.copy_(self.W_value_tensor[:, :d_out].T)
            
        self.dropout = torch.nn.Dropout(dropout)  # 드롭아웃 확률
        
        # 미래의 단어를 마스킹하는 상삼각 행렬 생성
        # register_buffer 메서드에 지정한 텐서를 모델과 함께 적절한 장치로 자동 이동시킨다.
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
        
    def forward(self, x):
        # x의 크기: (batch, num_tokens, d_in)     
        b, num_tokens, d_in = x.shape
        # 입력 x에 대해 쿼리, 키, 값 벡터 계산
        queries = self.W_query(x) # 모든 입력 토큰에 대한 쿼리 벡터 계산
        keys = self.W_key(x)     # 모든 입력 토큰에 대한 키 벡터 계산
        values = self.W_value(x)  # 모든 입력 토큰에 대한 값 벡터 계산
        
        # 첫 번재 차원인 배치 차원은 그대로 유지하면서 두 번재 차원과 세 번째 차원을 바꾼다.
        attn_scores = queries @ keys.transpose(1, 2) 
        
        # 마스크를 사용하여 어텐션 점수에서 주 대각선 위의 값을 -inf로 만듦
        #   - masked_fill: mask 행렬의 1인 위치에 -inf를 채움
        #   - "_"로 끝나는 메서드는 불필요한 메모리 복사를 피하기 위해 인플레이스 연산을 수행한다.
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], float('-inf'))

        attn_weights = torch.softmax(attn_scores / (keys.shape[-1] ** 0.5), dim=-1) # 어텐션 가중치 계산

        # 드롭아웃 적용
        attn_weights = self.dropout(attn_weights)
        
        context_vecs = attn_weights @ values # 문맥 벡터 계산
        return context_vecs

In [69]:
context_length = batch.shape[1] # 컨텍스트 길이
print("컨텍스트 길이: ", context_length)

# 입력 임베딩 크기 3, 출력 임베딩 크기 2
ca = CausalAttention(d_in=3, d_out=2, context_length=context_length, dropout=0.0) 

context_ves = ca(batch)
print("문맥 벡터:\n", context_ves)
print("문맥 벡터 크기: ", context_ves.shape)

컨텍스트 길이:  6
문맥 벡터:
 tensor([[[-0.2096,  0.4165],
         [-0.2737,  0.4614],
         [-0.2831,  0.4533],
         [-0.2701,  0.4432],
         [-0.2886,  0.4356],
         [-0.1316,  0.1758]],

        [[-0.2096,  0.4165],
         [-0.2737,  0.4614],
         [-0.2831,  0.4533],
         [-0.2701,  0.4432],
         [-0.2886,  0.4356],
         [-0.1316,  0.1758]]], grad_fn=<UnsafeViewBackward0>)
문맥 벡터 크기:  torch.Size([2, 6, 2])


## Chapter 7 싱글 헤드 어텐션을 멀티 헤드 어텐션으로 확장하기 <a class="anchor" id="chapter7"></a>
1. 멀티 헤드 어텐션은 싱글 헤드 어텐션을 확장한 것이다.
    - 각 어텐션 헤드는 고유한 가중치를 가진 셀프 어텐션 메커니즘을 여러 개 만들고 그 출력을 합친다.
    - 서로 다른 학습 가능한 싱글 헤드 어텐션을 병렬로 여러번 실행한다.

2. 2개의 싱글 헤드 어텐션 모듈을 쌓아서 구성
    - 두 개의 가중치 행렬을 사용하고 결과로 2개의 가중치 벡터가 생성된다.
    - 코잘 마스트, 드롭 아웃 마스크가 2개씩 필요하다.
    - 문맥 벡터도 2개가 생성된다. 

        ![멀티헤드 어텐션](image/03-06-mult2i.png)

In [70]:
# CauasalAttention 모듈을 여러 개 쌓아 간단한 MultiHeadAttentionWrapper 클래스 구현하기
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )
        self.out_proj = nn.Linear(d_out*num_heads, d_out*num_heads)

    def forward(self, x):
        context_vec = torch.cat([head(x) for head in self.heads], dim=-1)
        return self.out_proj(context_vec)

3. 2개의 헤드, d_out=2인 경우 문맥 벡트의 크기는 4가 된다.
    - 각각 문맥 벡터 행렬에서 행은 토큰에 대한 문맥 벡터이다.
    - 각 문맥 벡터의 열은 d_out=2로 지정한 임베딩 차원에 해당한다.
    - 문맥 벡터 행렬을 열차원에 따라 연결한다.

        ![문맥 차원](image/03-06-convec.png)

In [71]:
context_length = batch.shape[1] # 컨텍스트 길이 6

d_in, d_out = 3, 2 # 입력 임베딩 크기 3, 출력 임베딩 크기 2

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length=context_length, dropout=0.0, num_heads=2)

context_ves = mha(batch)
print("문맥 벡터:\n", context_ves)
print("문맥 벡터 크기: ", context_ves.shape)

문맥 벡터:
 tensor([[[ 0.1361, -0.1343,  0.3911, -0.4363],
         [ 0.1245, -0.1272,  0.4101, -0.4569],
         [ 0.1206, -0.1236,  0.4080, -0.4545],
         [ 0.1228, -0.1249,  0.4039, -0.4500],
         [ 0.1164, -0.1193,  0.4025, -0.4485],
         [ 0.1225, -0.1106,  0.3065, -0.3438]],

        [[ 0.1361, -0.1343,  0.3911, -0.4363],
         [ 0.1245, -0.1272,  0.4101, -0.4569],
         [ 0.1206, -0.1236,  0.4080, -0.4545],
         [ 0.1228, -0.1249,  0.4039, -0.4500],
         [ 0.1164, -0.1193,  0.4025, -0.4485],
         [ 0.1225, -0.1106,  0.3065, -0.3438]]], grad_fn=<ViewBackward0>)
문맥 벡터 크기:  torch.Size([2, 6, 4])


In [72]:
# 연습문제 3-2
#   num_heads=2는 그대로 두고 출력된 문맥 벡터가 4차원이 아니라 2차원이 되도록 입력 매개변수를 바꾸세요.
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

batch = torch.stack((inputs, inputs), dim=0)

context_length = batch.shape[1]
d_in = 3
d_out = 1

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[ 0.3254, -0.0696],
         [ 0.3083, -0.0490],
         [ 0.3066, -0.0469],
         [ 0.3083, -0.0490],
         [ 0.3116, -0.0529],
         [ 0.3382, -0.0849]],

        [[ 0.3254, -0.0696],
         [ 0.3083, -0.0490],
         [ 0.3066, -0.0469],
         [ 0.3083, -0.0490],
         [ 0.3116, -0.0529],
         [ 0.3382, -0.0849]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])


4. MultiHeadAttentionWrapper, CausalAttention 클래스를 하나의 클래스로 결합니다.
    - 선형 투형된 쿼리, 키, 값 텐서의 크기를 변형하는 방법을 사용하여 입력을 여러개의 해드로 나눈다.
    - 어텐션을 계산 후 각 헤드의 결과를 결합한다.

In [90]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out은 num_heads로 나누어 떨어져야 합니다"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # 원하는 출력 차원에 맞도록 투영 차원을 낮춥니다.

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear 층을 사용해 헤드의 출력을 결합합니다.
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # `CausalAttention`과 마찬가지로, 입력의 `num_tokens`가 `context_length`를 넘는 경우 마스크 생성에서 오류가 발생합니다.
        # 실제로는 forward 메서드에 들어오기 전에 LLM이 입력이 `context_length`를
        # 넘지 않는지 확인하기 때문에 문제가 되지 않습니다.

        keys = self.W_key(x) # 크기: (b, num_tokens, d_out)
        print("keys1:", keys.shape)
        queries = self.W_query(x)
        print("queries:", queries.shape)
        values = self.W_value(x)

        # `num_heads` 차원을 추가함으로써 암묵적으로 행렬을 분할합니다.
        # 그다음 마지막 차원을 `num_heads`에 맞춰 채웁니다: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        print("keys2:", keys.shape)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # 전치: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        print("keys3:", keys.shape)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        keys = keys.transpose(2, 3)
        print("keys4:", keys.shape)
        print("queries.shape : ",queries.shape)
        # 코잘 마스크로 스케일드 점곱 어텐션(셀프 어텐션)을 계산합니다.
        attn_scores = queries @ keys  # 각 헤드에 대해 점곱을 수행합니다.

        # 마스크를 불리언 타입으로 만들고 토큰 개수로 마스크를 자릅니다.
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # 마스크를 사용해 어텐션 점수를 채웁니다.
        attn_scores.masked_fill_(mask_bool, -torch.inf)

        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # 크기: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2)

        # 헤드를 결합합니다. self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # 투영

        return context_vec
    

In [91]:
batch_size, num_tokens, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, 2)
context_vecs = mha(batch)
#print("문맥 벡터:\n", context_vecs)
#print("문맥 벡터 크기: ", context_vecs.shape)

keys1: torch.Size([2, 6, 2])
queries: torch.Size([2, 6, 2])
keys2: torch.Size([2, 6, 2, 1])
keys3: torch.Size([2, 2, 6, 1])
keys4: torch.Size([2, 2, 1, 6])
queries.shape :  torch.Size([2, 2, 6, 1])


In [ ]:
# (b, num_heads, num_tokens, head_dim) = (1, 2, 3, 4)
a = torch.tensor([[[[0.2745, 0.6584, 0.2775, 0.8573],
                    [0.8993, 0.0390, 0.9268, 0.7388],
                    [0.7179, 0.7058, 0.9156, 0.4340]],

                   [[0.0772, 0.3565, 0.1479, 0.5331],
                    [0.4066, 0.2318, 0.4545, 0.9737],
                    [0.4606, 0.5159, 0.4220, 0.5786]]]])

#print(a)
#print(a.shape) # [1, 2, 3, 4]
#print("")

#print(a.transpose(2, 3))
#print(a.transpose(2, 3).shape) # [1, 2, 4, 3]
#print("")

print(a @ a.transpose(2, 3)) # [1, 2, 3, 3]

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


In [ ]:
first_head = a[0, 0, :, :]
print("첫 번째 헤드:\n", first_head)
print("첫 번째 헤드.T:\n", first_head.T)
first_res = first_head @ first_head.T
print("첫 번째 헤드:\n", first_res)

second_head = a[0, 1, :, :]
second_res = second_head @ second_head.T
print("\n두 번째 헤드:\n", second_res)

첫 번째 헤드:
 tensor([[0.2745, 0.6584, 0.2775, 0.8573],
        [0.8993, 0.0390, 0.9268, 0.7388],
        [0.7179, 0.7058, 0.9156, 0.4340]])
첫 번째 헤드.T:
 tensor([[0.2745, 0.8993, 0.7179],
        [0.6584, 0.0390, 0.7058],
        [0.2775, 0.9268, 0.9156],
        [0.8573, 0.7388, 0.4340]])
첫 번째 헤드:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

두 번째 헤드:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


In [ ]:
import torch.nn as nn

class MultiHeadAttention2(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), "d_out은 num_heads로 나누어 떨어져야합니다."
        
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # 원하는 출력 차원에 맞도록 투영 차원을 낮춘다.
        
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
            
        self.out_proj = nn.Linear(d_out, d_out) # Liner층을 사용해 헤드의 출력을 결합     
            
        self.dropout = torch.nn.Dropout(dropout)  # 드롭아웃 확률
        
        # 미래의 단어를 마스킹하는 상삼각 행렬 생성
        # register_buffer 메서드에 지정한 텐서를 모델과 함께 적절한 장치로 자동 이동시킨다.
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))
        
    def forward(self, x):
        # x의 크기: (batch, num_tokens, d_in)     
        b, num_tokens, d_in = x.shape
        # 입력 x에 대해 쿼리, 키, 값 벡터 계산
        keys = self.W_key(x)     # 모든 입력 토큰에 대한 키 벡터 계산
        queries = self.W_query(x) # 모든 입력 토큰에 대한 쿼리 벡터 계산
        values = self.W_value(x)  # 모든 입력 토큰에 대한 값 벡터 계산
        
        # num_heads 차원을 추가하여 쿼리, 키, 값 벡터를 (배치 크기, 토큰 수, 헤드 수, 헤드 차원) 크기로 변환
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)

        # (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)
        
        # 각 해드에 대한 점곱을 수행한다.
        attn_scores = queries @ keys.transpose(2, 3) 
        
        # 마스크를 사용하여 어텐션 점수에서 주 대각선 위의 값을 -inf로 만듦
        #   - masked_fill: mask 행렬의 1인 위치에 -inf를 채움
        #   - "_"로 끝나는 메서드는 불필요한 메모리 복사를 피하기 위해 인플레이스 연산을 수행한다.
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens] # 토큰 개수로 마스크를 자른다.
        attn_scores.masked_fill_(mask_bool, float('-inf'))

        attn_weights = torch.softmax(attn_scores / (keys.shape[-1] ** 0.5), dim=-1) # 어텐션 가중치 계산

        # 드롭아웃 적용
        attn_weights = self.dropout(attn_weights)
        
        # 문맥 벡터 계산
        context_vecs = (attn_weights @ values).transpose(1, 2) 
        
        context_vecs = context_vecs.contiguous().view(b, num_tokens, self.d_out)

        #context_vecs = self.out_proj(context_vecs)
        return context_vecs


context_length = 1024
d_in, d_out = 768, 768
num_heads = 12

mha = MultiHeadAttention2(d_in, d_out, context_length, 0.0, num_heads)



# 행렬 전치 이해하기
1. 행렬 전치란, 주어진 행렬의 행과 열을 바꾸는 연산입니다. 예를 들어, 2x3 행렬 A가 있을 때, A의 전치 행렬은 3x2 행렬로 변환됩니다. 전치 행렬은 일반적으로 A^T로 표기합니다.
     - 전치 행렬의 예시:
          A = [[1, 2, 3],
               [4, 5, 6]]

          A^T = [[1, 4],
               [2, 5],
               [3, 6]]

In [ ]:
import torch

tmp_a = torch.tensor([[1, 2, 3],
                      [4, 5, 6]])
tmp_b = tmp_a.transpose(0, 1)
print(tmp_b)

a = torch.tensor([[[[1, 2, 3, 4],
                    [5, 6, 7, 8],
                    [9, 10, 11, 12]],
                   [[13, 14, 15, 16],
                    [17, 18, 19, 20],
                    [21, 22, 23, 24]]]])
print(a)
print(a.shape)

# (1, 2, 3, 4) -> (1, 2, 4, 3)
# 2번째와 3번째 차원을 바꾼다(0부터 시작).
# 3차원 3행, 4열 -> 4행, 3열
a = a.transpose(2, 3)
print(a)
print(a.shape)




tensor([[1, 4],
        [2, 5],
        [3, 6]])
tensor([[[[ 1,  2,  3,  4],
          [ 5,  6,  7,  8],
          [ 9, 10, 11, 12]],

         [[13, 14, 15, 16],
          [17, 18, 19, 20],
          [21, 22, 23, 24]]]])
torch.Size([1, 2, 3, 4])
tensor([[[[ 1,  5,  9],
          [ 2,  6, 10],
          [ 3,  7, 11],
          [ 4,  8, 12]],

         [[13, 17, 21],
          [14, 18, 22],
          [15, 19, 23],
          [16, 20, 24]]]])
torch.Size([1, 2, 4, 3])


2. 멀티해드 어텐션 헤드가 2개인 경우 이해하기
   - 쿼리 벡터 구할 때 가중치 행렬 2개를 동시에 사용해서 쿼리 결과의 차원을 2배로 늘린다.
   - 첫번째 [1, 2] 
      - 입력 sample 별로 "keys = self.W_key(x)" 한후 
      ![예시](image/03-06-example.png)